In [ ]:
%cd /content/inferbench/inferbench
!pip install -e . -q

%cd /content/inferbench/inferbench
!git pull

/content/inferbench/inferbench
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for inferbench (pyproject.toml) ... done
/content/inferbench/inferbench
Already up to date.


In [ ]:
import os, torch, sys

print("cwd:", os.getcwd())
print("pyproject:", os.path.exists("pyproject.toml"))
print("scripts:", os.path.exists("scripts/run_bench.py"))
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


cwd: /content/inferbench/inferbench
pyproject: True
scripts: True
GPU: True
Device: Tesla T4


In [ ]:
from huggingface_hub import login
login()  # paste token from https://huggingface.co/settings/tokens

In [ ]:
%cd /content/inferbench/inferbench
!git pull
!pkill -f "custom/server.py" || true

# Confirm the DynamicCache fix is present (should show commit 0ae5e2f or newer)
!git log --oneline -1

import time, subprocess, requests
time.sleep(5)

# Server output goes to a log file (notebook stdout doesn't work with Popen)
log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["python", "src/backends/custom/server.py",
     "--model", "google/gemma-2-2b-it",
     "--port", "8003",
     "--max-batch-size", "4",
     "--kv-cache-blocks", "64"],
    stdout=log, stderr=subprocess.STDOUT,
)

print("Waiting for model to load (1-3 min)...")
ready = False
for i in range(90):
    try:
        if requests.get("http://127.0.0.1:8003/v1/health", timeout=2).status_code == 200:
            print(f"Engine ready after ~{i*2}s")
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    print("SERVER FAILED — log below:")
    print(open("/content/server.log").read())
    raise RuntimeError("server did not start")

# Smoke test — should finish in seconds now
t0 = time.time()
r = requests.post("http://127.0.0.1:8003/v1/completions",
    json={"model": "google/gemma-2-2b-it", "prompt": "2+2=", "max_tokens": 5, "temperature": 0.0},
    timeout=120)
print(f"Test took {time.time()-t0:.1f}s:", r.json()["choices"][0]["text"])

# Benchmark
!python scripts/run_bench.py \
    --backend custom \
    --model google/gemma-2-2b-it \
    --url http://127.0.0.1:8003 \
    --concurrency 4 \
    --duration 120 \
    --warmup 2 \
    --prompts prompts/short.jsonl \
    --output benchmarks/custom_gemma2_c4_short.json

/content/inferbench/inferbench
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.29 KiB | 658.00 KiB/s, done.
From https://github.com/preethamdandu/inferbench
   0ae5e2f..d6ce01c  main       -> origin/main
Updating 0ae5e2f..d6ce01c
Fast-forward
 scripts/smoke_engine.py       | 34 ++++++++++++++++++++++++++++++++--
 src/backends/custom/server.py |  2 +-
 2 files changed, 33 insertions(+), 3 deletions(-)
^C
d6ce01c (HEAD -> main, origin/main, origin/HEAD) fix: correct timing key in completion response
Waiting for model to load (1-3 min)...
Engine ready after ~18s
Test took 0.8s: 4

2+2
Loaded 50 prompts from prompts/short.jsonl
[custom] warming up (2 requests)...
[custom] benchmarking 4 concurrent workers for 120s...

[custom] RESULTS saved to benchmarks/custom_gemma2_c4_short.json
  throughput:  47.5 tok/

In [ ]:
%cd /content/inferbench/inferbench
!git pull

# 1. Custom benchmark (server still running from Cell 3) — exact token counts now
!python scripts/run_bench.py \
    --backend custom \
    --model google/gemma-2-2b-it \
    --url http://127.0.0.1:8003 \
    --concurrency 4 \
    --duration 120 \
    --warmup 2 \
    --prompts prompts/short.jsonl \
    --output benchmarks/custom_gemma2_c4_short.json

# 2. Free the GPU, then transformers on the SAME prompts (fair baseline)
!pkill -f "custom/server.py" || true
import time; time.sleep(5)

!python scripts/run_bench.py \
    --backend transformers \
    --model google/gemma-2-2b-it \
    --concurrency 1 \
    --duration 120 \
    --warmup 2 \
    --prompts prompts/short.jsonl \
    --output benchmarks/transformers_gemma2_c1_short.json

# 3. Report
!python -m src.bench.report
!cat benchmarks/report.md

/content/inferbench/inferbench
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 550 bytes | 275.00 KiB/s, done.
From https://github.com/preethamdandu/inferbench
   d6ce01c..3e14639  main       -> origin/main
Updating d6ce01c..3e14639
Fast-forward
 scripts/run_bench.py | 6 +++++-
 1 file changed, 5 insertions(+), 1 deletion(-)
Loaded 50 prompts from prompts/short.jsonl
[custom] warming up (2 requests)...
[custom] benchmarking 4 concurrent workers for 120s...

[custom] RESULTS saved to benchmarks/custom_gemma2_c4_short.json
  throughput:  69.3 tok/s
  latency P50: 2652 ms
  latency P99: 4027 ms
  TTFT P50:    96 ms
  error rate:  0.0%
^C
Loaded 50 prompts from prompts/short.jsonl
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install n

In [ ]:
!cat benchmarks/transformers_gemma2_c1.json
!python -m src.bench.report
!cat benchmarks/report.md

In [ ]:
!zip -r benchmarks.zip benchmarks/
from google.colab import files
files.download("benchmarks.zip")

updating: benchmarks/ (stored 0%)
updating: benchmarks/.gitkeep (stored 0%)
updating: benchmarks/sdxl_skipped.json (deflated 24%)
updating: benchmarks/tgi_fp16.json (deflated 22%)
updating: benchmarks/report.md (deflated 63%)
updating: benchmarks/vllm_awq.json (deflated 21%)
updating: benchmarks/transformers_gemma2_c1.json (deflated 55%)
  adding: benchmarks/transformers_gemma2_c1_short.json (deflated 55%)
  adding: benchmarks/custom_gemma2_c4_short.json (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content/inferbench/inferbench
!git fetch origin && git reset --hard origin/main
!python scripts/run_kernel_bench.py

/content/inferbench/inferbench
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 846 bytes | 211.00 KiB/s, done.
From https://github.com/preethamdandu/inferbench
   91ac5c0..42c0d9c  main       -> origin/main
HEAD is now at 42c0d9c kernel: replace try/except in GELU kernel with portable exp-based tanh
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
GPU: Tesla T4
Peak memory bandwidth: 320 GB/s

CORRECTNESS CHECK
  softmax (256, 1024): max_diff=7.45e-09  PASS
  softmax (512, 2048): max_diff=3.73e-09  PASS
  softmax (1024, 4096): max_diff=1.86e-09

In [ ]:
from google.colab import files
files.download("benchmarks/triton_kernels.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

vllm


In [ ]:
!pip uninstall -y -q vllm
!pip install -q "vllm==0.22.0" \
  --extra-index-url https://wheels.vllm.ai/0.22.0/cu129 \
  --extra-index-url https://download.pytorch.org/whl/cu129

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.0/473.0 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.9/360.9 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.4 MB/s eta 0:00:00


In [ ]:
!python -c "import vllm; print('vLLM OK:', vllm.__version__)"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
vLLM OK: 0.22.0


In [ ]:
import subprocess, time, requests

subprocess.run(["pkill", "-f", "vllm"], capture_output=True)
time.sleep(5)

log = open("/content/vllm.log", "w")
subprocess.Popen(
    ["vllm", "serve", "google/gemma-2-2b-it", "--port", "8001",
     "--dtype", "float32", "--max-model-len", "1024",
     "--gpu-memory-utilization", "0.92",
     "--enforce-eager", "--max-num-seqs", "16"],
    stdout=log, stderr=log, cwd="/content/inferbench/inferbench",
)

for i in range(120):
    try:
        if requests.get("http://localhost:8001/health", timeout=2).status_code == 200:
            print(f"vLLM ready after ~{i*5}s"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print("Not ready — run the grep cell again")

vLLM ready after ~110s


In [ ]:
!tail -50 /content/vllm.log


(APIServer pid=41579)     return self._loop.run_until_complete(task)
(APIServer pid=41579)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=41579)   File "uvloop/loop.pyx", line 1518, in uvloop.loop.Loop.run_until_complete
(APIServer pid=41579)   File "/usr/local/lib/python3.12/dist-packages/uvloop/__init__.py", line 48, in wrapper
(APIServer pid=41579)     return await main
(APIServer pid=41579)            ^^^^^^^^^^
(APIServer pid=41579)   File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py", line 678, in run_server
(APIServer pid=41579)     await run_server_worker(listen_address, sock, args, **uvicorn_kwargs)
(APIServer pid=41579)   File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py", line 692, in run_server_worker
(APIServer pid=41579)     async with build_async_engine_client(
(APIServer pid=41579)                ^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=41579)   File "/usr/lib/python3.12/contextlib.py", 

In [ ]:
!grep -n -i -B2 -A12 "EngineCore failed\|ValueError\|RuntimeError\|not supported\|out of memory\|capability" /content/vllm.log | head -100

21-(EngineCore pid=41749) INFO 07-06 08:09:23 [topk_topp_sampler.py:45] Using FlashInfer for top-p & top-k sampling.
22-(EngineCore pid=41749) INFO 07-06 08:09:23 [gpu_model_runner.py:5037] Starting to load model google/gemma-2-2b-it...
23:(EngineCore pid=41749) ERROR 07-06 08:09:24 [fa_utils.py:171] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
24-(EngineCore pid=41749) INFO 07-06 08:09:24 [cuda.py:378] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
25-(EngineCore pid=41749) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
26-(EngineCore pid=41749) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.n

In [ ]:
!pkill -f "vllm" || true
!pkill -f "server.py" || true
import time; time.sleep(5)
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

^C
^C
pid, process_name, used_gpu_memory [MiB]
10730, python3, 5484 MiB
memory.used [MiB], memory.total [MiB]
5487 MiB, 15360 MiB


In [ ]:
import os
print(os.getpid())


18790


In [ ]:
!kill -9 10730
import time; time.sleep(3)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB


In [ ]:
%cd /content/inferbench/inferbench
!python scripts/run_bench.py --backend vllm --model google/gemma-2-2b-it \
  --concurrency 4 --duration 120 --warmup 2 \
  --prompts prompts/short.jsonl --output benchmarks/vllm_gemma2_c4_short.json

Streaming output truncated to the last 5000 lines.
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call failed ('127.0.0.1', 8001)]
  [warn] request failed: ClientConnectorError: Cannot connect to host localhost:8001 ssl:default [Connect call fail

In [ ]:
!pgrep -af vllm || echo "vllm process is GONE"
!tail -40 /content/vllm.log


48243 /bin/bash -c pgrep -af vllm || echo "vllm process is GONE"
(APIServer pid=44962) INFO:     127.0.0.1:43036 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43036 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43016 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43024 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43012 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43036 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43016 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43012 - "POST /v1/completions HTTP/1.1" 500 Internal Server Error
(APIServer pid=44962) INFO:     127.0.0.1:43024 - "POST /v1/completions

In [ ]:
!grep -n -i -B3 -A15 "CUDA out of memory\|EngineDead\|ERROR.*core\|Traceback" /content/vllm.log | head -120

88-(EngineCore pid=45140) ERROR 07-06 08:27:17 [dump_input.py:72] Dumping input data for V1 LLM engine (v0.22.0) with config: model='google/gemma-2-2b-it', speculative_config=None, tokenizer='google/gemma-2-2b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float32, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp

In [ ]:
!pkill -f "vllm serve" || true
!cat benchmarks/vllm_gemma2_c4_short.json